In [10]:

import pandas as pd
import plotly.express as px
import numpy as np

# Load dataset from Kaggle
df = pd.read_csv(
    '/kaggle/input/datasets/josephsudheer0506/world-population/world_population.csv'
)

# Shorter column names
df = df.rename(columns={
    'Country/Territory': 'Country',
    'Area (km²)': 'Area',
    'Density (per km²)': 'Density',
    'World Population Percentage': 'World share (%)'
})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]

df_long = df.melt(
    id_vars=['CCA3', 'Country', 'Continent'],
    value_vars=year_cols,
    var_name='Year',
    value_name='Population'
)

df_long['Year'] = df_long['Year'].str[:4].astype(int)

print("Original dataset:", df.shape)
print("Long dataset:", df_long.shape)

display(df.head())
display(df_long.head())


Original dataset: (234, 16)
Long dataset: (1872, 5)


,Rank,CCA3,Country,Capital,Continent,2022 Population,2020 Population,2015 Population,2010 Population,2000 Population,1990 Population,1980 Population,1970 Population,Area,Density,World share (%)
0,36,AFG,Afghanistan,Kabul,Asia,41128771,38972230,33753499,28189672,19542982,10694796,12486631,10752971,652230,63.0587,0.52
1,138,ALB,Albania,Tirana,Europe,2842321,2866849,2882481,2913399,3182021,3295066,2941651,2324731,28748,98.8702,0.04
2,34,DZA,Algeria,Algiers,Africa,44903225,43451666,39543154,35856344,30774621,25518074,18739378,13795915,2381741,18.8531,0.56
3,213,ASM,American Samoa,Pago Pago,Oceania,44273,46189,51368,54849,58230,47818,32886,27075,199,222.4774,0.00
4,203,AND,Andorra,Andorra la Vella,Europe,79824,77700,71746,71519,66097,53569,35611,19860,468,170.5641,0.00


,CCA3,Country,Continent,Year,Population
0,AFG,Afghanistan,Asia,2022,41128771
1,ALB,Albania,Europe,2022,2842321
2,DZA,Algeria,Africa,2022,44903225
3,ASM,American Samoa,Oceania,2022,44273
4,AND,Andorra,Europe,2022,79824


In [12]:

# Q1 — Which continent is growing fastest?

# Aggregate population by continent and year
continent_pop = (
    df_long.groupby(['Continent', 'Year'])['Population']
    .sum()
    .reset_index()
)

# Calculate relative growth (1970 = 100)
baseline = (
    continent_pop[continent_pop['Year'] == 1970]
    [['Continent', 'Population']]
    .rename(columns={'Population': 'Population_1970'})
)

continent_pop = continent_pop.merge(baseline, on='Continent')

continent_pop['Growth_Index'] = (
    continent_pop['Population'] /
    continent_pop['Population_1970'] * 100
)

# Find fastest-growing continent in the latest year
latest_year = continent_pop['Year'].max()

ranking = (
    continent_pop[continent_pop['Year'] == latest_year]
    .sort_values('Growth_Index', ascending=False)
)

fastest = ranking.iloc[0]['Continent']
growth = ranking.iloc[0]['Growth_Index']

print("Population growth ranking (1970 = 100):")
print(ranking[['Continent', 'Growth_Index']].round(1).to_string(index=False))
print(f"\nFastest-growing continent: {fastest}")
print(f"Population is {growth/100:.2f} times its 1970 size.")

# Highlight fastest-growing continent
continent_pop['Group'] = continent_pop['Continent'].apply(
    lambda x: 'Fastest-growing' if x == fastest else 'Other continents'
)

# Create line chart
fig = px.line(
    continent_pop,
    x='Year',
    y='Growth_Index',
    color='Group',
    line_group='Continent',
    color_discrete_map={
        'Fastest-growing': '#2563EB',
        'Other continents': '#D1D5DB'
    },
    title=f"<b>{fastest} has grown fastest since 1970</b>",
    labels={
        'Year': 'Year',
        'Growth_Index': 'Population index (1970 = 100)',
        'Group': ''
    }
)

# Highlight the main line
fig.update_traces(
    line=dict(width=2),
    selector=dict(name='Other continents')
)

fig.update_traces(
    line=dict(width=3.5),
    selector=dict(name='Fastest-growing')
)

# Direct label at the end of highlighted line
fig.add_annotation(
    x=latest_year,
    y=growth,
    text=f"<b>{fastest}</b>",
    showarrow=False,
    xshift=35,
    font=dict(color='#2563EB', size=13)
)

fig.update_layout(
    template='plotly_white',
    plot_bgcolor='white',
    title_x=0.02,
    showlegend=False,
    margin=dict(l=65, r=110, t=85, b=55)
)

fig.update_xaxes(showgrid=False, showline=False)
fig.update_yaxes(
    showgrid=True,
    gridcolor='#EEEEEE',
    showline=False,
    rangemode='tozero'
)

fig.show()


Population growth ranking (1970 = 100):
    Continent  Growth_Index
       Africa         390.4
      Oceania         231.2
South America         226.4
         Asia         220.1
North America         190.3
       Europe         113.3

Fastest-growing continent: Africa
Population is 3.90 times its 1970 size.




I chose a line chart because it clearly shows how population growth changes over time across different continents. I indexed each continent's population to its 1970 population (1970 = 100), allowing a fair comparison of relative growth regardless of initial population size.

I used grey lines for context and highlighted the fastest-growing continent in blue. This makes the main finding easy to identify without unnecessary visual clutter.

In [14]:

# Q2 — Which countries added the most people?

# Calculate population increase between 1970 and 2022
df['People_Added'] = (
    df['2022 Population'] - df['1970 Population']
)

# Select top 15 countries
top15 = (
    df.nlargest(15, 'People_Added')
    .copy()
)

# Count African countries
africa_count = (top15['Continent'] == 'Africa').sum()

print("Top 15 countries that added the most people:")
print(
    top15[['Country', 'People_Added', 'Continent']]
    .sort_values('People_Added', ascending=False)
    .to_string(index=False)
)

print(f"\nNumber of African countries in the top 15: {africa_count}")

# Highlight African countries
top15['Group'] = top15['Continent'].apply(
    lambda x: 'Africa' if x == 'Africa' else 'Other continents'
)

# Sort for horizontal bar chart
top15 = top15.sort_values('People_Added', ascending=True)

# Create chart
fig = px.bar(
    top15,
    x='People_Added',
    y='Country',
    color='Group',
    orientation='h',
    text='People_Added',
    color_discrete_map={
        'Africa': '#2563EB',
        'Other continents': '#D1D5DB'
    },
    title=f"<b>Top 15 population gainers: {africa_count} African countries</b>",
    labels={
        'People_Added': 'People added (1970–2022)',
        'Country': '',
        'Group': ''
    }
)

# Format labels
fig.update_traces(
    texttemplate='%{text:,.0f}',
    textposition='outside',
    cliponaxis=False
)

# Clean layout
fig.update_layout(
    template='plotly_white',
    showlegend=False,
    title_x=0.02,
    plot_bgcolor='white',
    margin=dict(l=110, r=100, t=85, b=55)
)

fig.update_xaxes(
    showgrid=True,
    gridcolor='#EEEEEE',
    showline=False,
    tickformat=','
)

fig.update_yaxes(
    showgrid=False,
    showline=False
)
fig.update_yaxes(categoryorder='total ascending')
fig.show()


Top 15 countries that added the most people:
      Country  People_Added     Continent
        India     859671872          Asia
        China     603352887          Asia
     Pakistan     176533990          Asia
      Nigeria     162971948        Africa
    Indonesia     160272945          Asia
United States     137961517 North America
       Brazil     118943623 South America
   Bangladesh     103644512          Asia
     Ethiopia      95071678        Africa
     DR Congo      78858479        Africa
  Philippines      78123423          Asia
       Mexico      77214819 North America
        Egypt      76208117        Africa
         Iran      60100865          Asia
      Vietnam      56258007          Asia

Number of African countries in the top 15: 4


Chart choice & justification: Sorted horizontal bar chart

I chose a sorted horizontal bar chart because it makes it easy to compare the population increase across the 15 countries. I calculated population growth by subtracting the 1970 population from the 2022 population. I used grey for countries outside Africa and blue to highlight African countries, making it easy to identify how many African countries appear in the top 15. The bars are sorted to make the ranking clear.

Key finding:

India added the most people between 1970 and 2022, with approximately 859.7 million additional people, followed by China with 603.4 million. Out of the top 15 countries, 4 are in Africa: Nigeria, Ethiopia, DR Congo and Egypt.

In [15]:

# Q3 — Do bigger countries have more people?

# Prepare data
plot_df = df[
    ['Country', 'Continent', 'Area', '2022 Population']
].dropna().copy()

# Remove invalid values for logarithmic scales
plot_df = plot_df[
    (plot_df['Area'] > 0) &
    (plot_df['2022 Population'] > 0)
].copy()

# Calculate log values
plot_df['Log_Area'] = np.log10(plot_df['Area'])
plot_df['Log_Population'] = np.log10(plot_df['2022 Population'])

# Find countries that deviate from the general relationship
slope, intercept = np.polyfit(
    plot_df['Log_Area'],
    plot_df['Log_Population'],
    1
)

plot_df['Residual'] = (
    plot_df['Log_Population'] -
    (slope * plot_df['Log_Area'] + intercept)
)

# Label the 3 countries above and 3 below the trend
outliers = pd.concat([
    plot_df.nlargest(3, 'Residual'),
    plot_df.nsmallest(3, 'Residual')
])

plot_df['Label'] = plot_df['Country'].where(
    plot_df['Country'].isin(outliers['Country']), ''
)

print("Countries above the general pattern:")
print(plot_df.nlargest(3, 'Residual')['Country'].tolist())

print("\nCountries below the general pattern:")
print(plot_df.nsmallest(3, 'Residual')['Country'].tolist())

# Create scatter plot
fig = px.scatter(
    plot_df,
    x='Area',
    y='2022 Population',
    color='Continent',
    text='Label',
    hover_name='Country',
    hover_data={
        'Area': ':,.0f',
        '2022 Population': ':,.0f',
        'Continent': True
    },
    color_discrete_sequence=px.colors.qualitative.Safe,
    title='<b>Land area alone does not explain a country’s population</b>',
    labels={
        'Area': 'Land area (km²)',
        '2022 Population': 'Population (2022)',
        'Continent': 'Continent'
    }
)

# Logarithmic scales
fig.update_xaxes(type='log', showgrid=False)
fig.update_yaxes(
    type='log',
    showgrid=True,
    gridcolor='#EEEEEE'
)

# Direct labels for unusual countries
fig.update_traces(
    marker=dict(size=10, opacity=0.8),
    textposition='top center',
    textfont=dict(size=10)
)

# Clean design
fig.update_layout(
    template='plotly_white',
    plot_bgcolor='white',
    title_x=0.02,
    legend_title_text='Continent',
    margin=dict(l=75, r=40, t=90, b=65)
)

fig.show()


Countries above the general pattern:
['Macau', 'Singapore', 'Hong Kong']

Countries below the general pattern:
['Greenland', 'Falkland Islands', 'Niue']


Chart choice & justification: Scatter plot

I chose a scatter plot because it helps identify the relationship between a country's land area and its population. I used land area on the X-axis and 2022 population on the Y-axis, with logarithmic scales to make it easier to compare countries of different sizes. I used different CVD-safe colours for each continent to identify regional differences. I also labelled countries that clearly deviate from the general pattern to highlight unusual cases.

Key finding:

There is a positive relationship between land area and population, meaning larger countries generally tend to have more people. However, land area alone does not determine population size. Hong Kong and Macau have relatively high populations despite their small land areas, while Greenland, Falkland Islands and Niue have very low populations relative to their land areas. The pattern also varies across continents, showing that population distribution differs significantly between regions.

In [19]:

# Q4 — How crowded are the continents?

# Prepare data
density_df = df[
    ['Country', 'Continent', 'Density']
].dropna().copy()

# Remove invalid density values
density_df = density_df[density_df['Density'] > 0]

# Calculate median density by continent
medians = (
    density_df.groupby('Continent')['Density']
    .median()
    .sort_values(ascending=False)
)

print("Median population density by continent:")
print(medians.round(1))

# Find continent with highest median density
highest = medians.index[0]

print(f"\nHighest median density: {highest}")

# Highlight highest median continent
density_df['Group'] = density_df['Continent'].apply(
    lambda x: 'Highest median' if x == highest else 'Other continents'
)

# Create box plot
fig = px.box(
    density_df,
    x='Continent',
    y='Density',
    color='Group',
    points='outliers',
    color_discrete_map={
        'Highest median': '#2563EB',
        'Other continents': '#D1D5DB'
    },
    title=f"<b>{highest} has the highest median population density</b>",
    labels={
        'Continent': '',
        'Density': 'Population density (people per km²)',
        'Group': ''
    },
    hover_name='Country'
)

# Logarithmic scale
fig.update_yaxes(
    type='log',
    showgrid=True,
    gridcolor='#EEEEEE'
)

# Clean layout
fig.update_layout(
    template='plotly_white',
    plot_bgcolor='white',
    title_x=0.02,
    showlegend=False,
    margin=dict(l=75, r=35, t=85, b=65)
)

fig.update_xaxes(showgrid=False, showline=False)
category_orders={
    'Continent': medians.index.tolist()
}

fig.show()


Median population density by continent:
Continent
North America    221.4
Asia             123.0
Europe           106.9
Oceania           78.2
Africa            59.7
South America     17.8
Name: Density, dtype: float64

Highest median density: North America


Chart choice & justification: Box plot

I chose a box plot because it shows the distribution and spread of population density across different continents. It displays the median, interquartile range and outliers, making it easier to compare how population density varies within and between continents. I used a grey colour for most continents and blue to highlight the continent with the highest median density. I also used a logarithmic scale to make differences between highly and sparsely populated regions easier to see.

Key finding:

North America has the highest median population density at 221.4 people per km², followed by Asia at 123.0 and Europe at 106.9. South America has the lowest median density at 17.8 people per km². The box plot shows that population density varies considerably within each continent, with several high-density outliers. This demonstrates that population density is not evenly distributed within or across continents.

In [20]:

# Q5 — Where did populations grow and shrink?

# Calculate percentage population change (2000–2022)
map_df = df[
    ['Country', 'CCA3', 'Continent',
     '2000 Population', '2022 Population']
].dropna().copy()

map_df = map_df[map_df['2000 Population'] > 0].copy()

map_df['Change_pct'] = (
    (map_df['2022 Population'] - map_df['2000 Population'])
    / map_df['2000 Population']
) * 100

# Identify growth and decline
map_df['Status'] = np.where(
    map_df['Change_pct'] >= 0,
    'Population growth',
    'Population decline'
)

# Print summary
growth_count = (map_df['Change_pct'] > 0).sum()
decline_count = (map_df['Change_pct'] < 0).sum()

print("Countries with population growth:", growth_count)
print("Countries with population decline:", decline_count)

print("\nTop 10 countries with highest population growth:")
print(
    map_df.nlargest(10, 'Change_pct')
    [['Country', 'Continent', 'Change_pct']]
    .round(1).to_string(index=False)
)

print("\nTop 10 countries with population decline:")
print(
    map_df.nsmallest(10, 'Change_pct')
    [['Country', 'Continent', 'Change_pct']]
    .round(1).to_string(index=False)
)

# Create choropleth map
limit = map_df['Change_pct'].abs().max()

fig = px.choropleth(
    map_df,
    locations='CCA3',
    locationmode='ISO-3',
    color='Change_pct',
    hover_name='Country',
    hover_data={
        'Continent': True,
        '2000 Population': ':,',
        '2022 Population': ':,',
        'Change_pct': ':.1f',
        'CCA3': False
    },
    color_continuous_scale=[
        [0, '#2166AC'],
        [0.5, '#F7F7F7'],
        [1, '#B35806']
    ],
    range_color=[-limit, limit],
    title='<b>Population growth and decline vary across the world (2000–2022)</b>',
    labels={
        'Change_pct': 'Population change (%)'
    }
)

# Clean map design
fig.update_geos(
    projection_type='natural earth',
    showframe=False,
    showcoastlines=True,
    coastlinecolor='#AAAAAA',
    showcountries=True,
    countrycolor='white',
    showland=True,
    landcolor='#EEEEEE'
)

fig.update_layout(
    template='plotly_white',
    title_x=0.02,
    margin=dict(l=20, r=20, t=85, b=30),
    coloraxis_colorbar=dict(
        title='Change (%)'
    )
)

fig.show()


Countries with population growth: 200
Countries with population decline: 34

Top 10 countries with highest population growth:
                 Country     Continent  Change_pct
                   Qatar          Asia       317.2
    United Arab Emirates          Asia       188.2
       Equatorial Guinea        Africa       144.5
Turks and Caicos Islands North America       143.8
                   Niger        Africa       125.5
                  Jordan          Asia       123.2
                  Kuwait          Asia       120.6
                  Angola        Africa       117.1
                    Chad        Africa       114.6
          Western Sahara        Africa       113.0

Top 10 countries with population decline:
                 Country Continent  Change_pct
Northern Mariana Islands   Oceania       -38.3
          American Samoa   Oceania       -24.0
               Lithuania    Europe       -23.6
        Marshall Islands   Oceania       -23.3
                 Moldova    Europe 

Chart choice & justification: Choropleth map

I chose a choropleth map because it clearly shows the geographical distribution of population growth and decline across countries. I calculated the percentage change in population between 2000 and 2022. I used a diverging colour scale, with blue representing population growth and orange representing population decline. This makes it easy to identify geographical patterns and compare population changes across the world.

Key findings:

Between 2000 and 2022, 200 countries experienced population growth, while 34 experienced population decline. Qatar recorded the highest population growth at 317.2%, followed by the United Arab Emirates at 188.2%. The Northern Mariana Islands experienced the largest decline at -38.3%.

The choropleth map shows geographical differences in population change. Strong population growth is particularly visible in parts of Africa and Asia, while several European and Pacific island countries experienced population declines. Overall, population growth was widespread, but the rate of change varied considerably across countries.

In [24]:

# Q6 — Where do the world's 8 billion people live?

# Calculate total population by continent
continent_totals = (
    df.groupby('Continent')['2022 Population']
    .sum()
    .sort_values(ascending=False)
)

print("2022 Population by continent:")
print(continent_totals.apply(lambda x: f"{x:,.0f}"))

print("\nPercentage of world population:")
print((continent_totals / continent_totals.sum() * 100).round(1))

# Create treemap
fig = px.treemap(
    df,
    path=[px.Constant('World'), 'Continent', 'Country'],
    values='2022 Population',
    color='Continent',
    color_discrete_sequence=[
        '#4477AA',
        '#EEAA33',
        '#228833',
        '#AA3377',
        '#66CCEE',
        '#BBBBBB'
    ],
    hover_data={
        '2022 Population': ':,'
    },
    title="<b>Asia contains the largest share of the world's population</b>"
)

# Improve labels and layout
fig.update_traces(
    textinfo='label+percent parent',
    textfont_size=12,
    marker=dict(line=dict(color='white', width=2))
)

fig.update_layout(
    template='plotly_white',
    title_x=0.02,
    margin=dict(t=80, l=20, r=20, b=20),
    showlegend=False
)

fig.show()


2022 Population by continent:
Continent
Asia             4,721,383,274
Africa           1,426,730,932
Europe             743,147,538
North America      600,296,136
South America      436,816,608
Oceania             45,038,554
Name: 2022 Population, dtype: object

Percentage of world population:
Continent
Asia             59.2
Africa           17.9
Europe            9.3
North America     7.5
South America     5.5
Oceania           0.6
Name: 2022 Population, dtype: float64


Chart choice & justification: Treemap

I chose a treemap because it displays hierarchical data at two levels: continents and individual countries. Each continent is divided into rectangles representing its countries, with the rectangle size proportional to the 2022 population. I used different accessible colours for each continent to make the groups easy to distinguish. This visualization makes it easy to compare population distribution across continents and identify the countries with the largest populations.


Key findings:

Asia has the largest share of the world's population, with approximately 4.72 billion people, representing 59.2% of the global population. Africa is the second-largest continent, with 1.43 billion people (17.9%), followed by Europe with 743.1 million (9.3%). Oceania has the smallest population, with approximately 45 million people (0.6%).

At the country level, India and China have the largest population rectangles within Asia, showing their significant contribution to the global population. Nigeria is a major contributor in Africa, while the United States dominates North America. Overall, the treemap demonstrates that the world's population is heavily concentrated in Asia.

In [26]:

# Q7 — Whose share of the world is rising?

# Group continents
df_q7 = df_long.copy()

df_q7['Group'] = df_q7['Continent'].where(
    df_q7['Continent'].isin(['Africa', 'Asia', 'Europe']),
    'Rest of the world'
)

# Calculate population by group and year
group_pop = (
    df_q7.groupby(['Year', 'Group'])['Population']
    .sum()
    .reset_index()
)

# Calculate total world population per year
world_pop = (
    group_pop.groupby('Year')['Population']
    .sum()
    .reset_index(name='World_Population')
)

# Calculate percentage share
group_pop = group_pop.merge(world_pop, on='Year')

group_pop['Share_pct'] = (
    group_pop['Population'] /
    group_pop['World_Population'] * 100
)

# Print 1970 and 2022 comparison
comparison = group_pop[
    group_pop['Year'].isin([1970, 2022])
].pivot(
    index='Group',
    columns='Year',
    values='Share_pct'
)

comparison['Change (percentage points)'] = (
    comparison[2022] - comparison[1970]
)

print("Population share comparison (1970–2022):")
print(comparison.round(2))

# Create line chart
fig = px.line(
    group_pop,
    x='Year',
    y='Share_pct',
    color='Group',
    markers=True,
    color_discrete_map={
        'Africa': '#0072B2',
        'Asia': '#E69F00',
        'Europe': '#777777',
        'Rest of the world': '#BBBBBB'
    },
    title="<b>Africa gains world population share while Europe's declines</b>",
    labels={
        'Year': 'Year',
        'Share_pct': 'Share of world population (%)',
        'Group': 'Region'
    }
)

fig.update_traces(line=dict(width=3), marker=dict(size=6))

fig.update_layout(
    template='plotly_white',
    title_x=0.02,
    margin=dict(l=50, r=30, t=80, b=50),
    legend_title_text='',
    yaxis=dict(showgrid=False, ticksuffix='%'),
    xaxis=dict(showgrid=False, dtick=10)
)

fig.show()


Population share comparison (1970–2022):
Year                1970   2022  Change (percentage points)
Group                                                      
Africa              9.89  17.89                        8.00
Asia               58.06  59.21                        1.15
Europe             17.76   9.32                       -8.44
Rest of the world  14.29  13.57                       -0.72


Chart choice & justification: Line chart

I chose a line chart because it clearly shows how each region's share of the world population changes over time. I calculated each continent's population as a percentage of the total world population for every year. I grouped North America, South America and Oceania into 'Rest of the world'. This makes it easy to compare population trends and identify which regions are gaining or losing their share.


Key findings:

Between 1970 and 2022, Africa experienced the largest increase in its share of the world's population, rising from 9.89% to 17.89%, an increase of 8 percentage points. Asia remained the largest contributor, with its share increasing slightly from 58.06% to 59.21%. In contrast, Europe's share declined significantly from 17.76% to 9.32%, a decrease of 8.44 percentage points. The Rest of the World also experienced a small decline of 0.72 percentage points.

Overall, Africa is gaining its share of the global population, while Europe's share is declining. Asia continues to account for the largest proportion of the world's population.

##### 